## Compare RNNs, LSTMs, and GRUs with Tensorflow

### Part1 - Why SimpleRNN needs help !

#### step 1 - Import Tensorflow and Set Up

In [4]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import tensorflow_datasets as tfds
from datasets import load_dataset, load_from_disk

In [2]:
try:
    import tensorflow_datasets as tfds
except ImportError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'tensorflow_datasets'])
    import tensorflow_datasets as tfds

In [3]:
tf.keras.utils.set_random_seed(42)
tf.__version__

'2.22.0-rc0'

#### Step 2 - Review What SimpleRNN Did

###### TextVectorization -> Embedding -> SimpleRNN -> Dense

#### Step 3 - The Memory Problem

###### Long sequences make memory and learning harder for a basic RNN.
###### This is one reason LSTMs and GRUs were created.

#### Step 4 - What LSTMs and GRUs Add
###### LSTMs and GRUs are still sequence models.
###### They still read tokens in order
###### The difference is that they add better memory control

### Part 2 - Prepare the Movie Review Dataset
#### Step 5 - Load the IMDB Dataset

In [5]:
# Load from local disk (no internet needed)
dataset = load_from_disk("./imdb_stanfordnlp")

train_hf = dataset["train"].shuffle(seed=42).select(range(10000))
test_hf  = dataset["test"].shuffle(seed=42).select(range(5000))

train_data = tf.data.Dataset.from_tensor_slices((
        list(train_hf["text"]),
        list(train_hf["label"])
    ))
test_data  = tf.data.Dataset.from_tensor_slices((
        list(test_hf["text"]),
        list(test_hf["label"])
    ))

print(train_data)
print(test_data)

<_TensorSliceDataset element_spec=(TensorSpec(shape=(), dtype=tf.string, name=None), TensorSpec(shape=(), dtype=tf.int32, name=None))>
<_TensorSliceDataset element_spec=(TensorSpec(shape=(), dtype=tf.string, name=None), TensorSpec(shape=(), dtype=tf.int32, name=None))>


#### Step 6 - Inspect a Few Review
###### Before we build model, we inspect the data.

In [6]:
for review, label in train_data.take(2):
    print("Review:")
    print(review.numpy().decode("utf-8")[:600], "...")
    print("Label: ", label.numpy())
    print("-" * 100)

Review:
There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier's plot are far more complicated... Fortier looks more like Prime Suspect, if we have to spot similarities... The main character is weak and weirdo, but have "clairvoyance". People like to compare, to judge, to evaluate. How about just enjoying? Funny thing too, people writing Fortier looks American but, on the other hand, arguing they prefer American series (!!!). Maybe it's the language, o ...
Label:  1
----------------------------------------------------------------------------------------------------
Review:
This movie is a great. The plot is very true to the book which is a classic written by Mark Twain. The movie starts of with a scene where Hank sings a song with a bunch of kids called "when you stub your toe on the moon" It reminds me of Sinatra's song High Hopes, it 

#### Step 7 - Create the TextVectorizer Layer

In [7]:
max_tokens = 10000
sequence_length = 150

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=sequence_length
)
train_text_only = train_data.map(lambda text, labels: text)
vectorizer.adapt(train_text_only.batch(1024))

vocab = vectorizer.get_vocabulary()
print(len(vocab))
print(vocab[:20])
print(vocab[-20:])

10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 84ms/step 
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 84ms/step
10000
['', '[UNK]', np.str_('the'), np.str_('and'), np.str_('a'), np.str_('of'), np.str_('to'), np.str_('is'), np.str_('in'), np.str_('it'), np.str_('i'), np.str_('this'), np.str_('that'), np.str_('br'), np.str_('was'), np.str_('as'), np.str_('with'), np.str_('for'), np.str_('but'), np.str_('movie')]
[np.str_('syrup'), np.str_('suspiciously'), np.str_('summarized'), np.str_('sullavan'), np.str_('sucking'), np.str_('sucker'), np.str_('successor'), np.str_('succeeding'), np.str_('suburbia'), np.str_('strain'), np.str_('stockwell'), np.str_('stifler'), np.str_('stepped'), np.str_('steamy'), np.str_('staden'), np.str_('stacy'), np.str_('squeamish'), np.str_('spitting'), np.str_('spins'), np.str_('spinning')]
